In [1]:
from pathlib import Path
import os
import requests
import pandas as pd
import xml.etree.ElementTree as ET

from dotenv import load_dotenv


# ============================================================
# 경로 / API 설정
# ============================================================

PROJECT_DIR = Path.cwd().parents[1]

DATA_ROOT = PROJECT_DIR / "data"
DATA_DIR = DATA_ROOT / "v2_5year_seoul"

RAW_DIR = DATA_DIR / "raw" / "rent"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

load_dotenv(
    PROJECT_DIR / ".env"
)

SERVICE_KEY = os.getenv(
    "MOLIT_RENT_API_KEY"
)

BASE_URL = (
    "https://apis.data.go.kr/1613000/"
    "RTMSDataSvcAptRent/"
    "getRTMSDataSvcAptRent"
)

if not SERVICE_KEY:
    raise ValueError(
        "MOLIT_RENT_API_KEY가 없습니다."
    )

In [2]:
# ============================================================
# API 단건 테스트
# ============================================================

params = {
    "serviceKey": SERVICE_KEY,
    "LAWD_CD": "11680",     # 강남구
    "DEAL_YMD": "202606",
    "pageNo": 1,
    "numOfRows": 100,
}

response = requests.get(
    BASE_URL,
    params=params,
    timeout=30,
)

print("status:", response.status_code)
print(response.text[:1000])

status: 200
<?xml version="1.0" encoding="utf-8" standalone="yes"?><response><header><resultCode>000</resultCode><resultMsg>OK</resultMsg></header><body><items><item><aptNm>아카데미스위트1</aptNm><aptSeq>11680-566</aptSeq><buildYear>2004</buildYear><contractTerm> </contractTerm><contractType> </contractType><dealDay>25</dealDay><dealMonth>6</dealMonth><dealYear>2026</dealYear><deposit>70,000</deposit><excluUseAr>122.07</excluUseAr><floor>7</floor><jibun>467-7</jibun><monthlyRent>230</monthlyRent><preDeposit> </preDeposit><preMonthlyRent> </preMonthlyRent><roadnm>언주로30길 21</roadnm><roadnmbcd></roadnmbcd><roadnmbonbun>00021</roadnmbonbun><roadnmbubun>00000</roadnmbubun><roadnmcd>4166604</roadnmcd><roadnmseq>0</roadnmseq><roadnmsggcd>11680</roadnmsggcd><sggCd>11680</sggCd><umdNm>도곡동</umdNm><useRRRight> </useRRRight></item><item><aptNm>개나리푸르지오</aptNm><aptSeq>11680-3678</aptSeq><buildYear>2006</buildYear><contractTerm> </contractTerm><contractType> </contractType><dealDay>13</dealDay><dealMonth>6<

In [3]:
import xml.etree.ElementTree as ET


def parse_rent_xml(content):
    root = ET.fromstring(content)

    result_code = root.findtext(".//resultCode")
    result_msg = root.findtext(".//resultMsg")

    if result_code != "000":
        raise ValueError(
            f"API 오류: {result_code} / {result_msg}"
        )

    rows = []

    for item in root.findall(".//item"):
        row = {
            child.tag: (
                child.text.strip()
                if child.text
                else None
            )
            for child in item
        }

        rows.append(row)

    return pd.DataFrame(rows)


test_df = parse_rent_xml(
    response.content
)

print("조회 건수:", len(test_df))
print("컬럼 수:", len(test_df.columns))

display(test_df.head())

print(test_df.columns.tolist())

조회 건수: 100
컬럼 수: 25


,aptNm,aptSeq,buildYear,contractTerm,contractType,dealDay,dealMonth,dealYear,deposit,excluUseAr,...,roadnm,roadnmbcd,roadnmbonbun,roadnmbubun,roadnmcd,roadnmseq,roadnmsggcd,sggCd,umdNm,useRRRight
0,아카데미스위트1,11680-566,2004,,,25,6,2026,"70,000",122.07,...,언주로30길 21,None,00021,00000,4166604,0,11680,11680,도곡동,
1,개나리푸르지오,11680-3678,2006,,,13,6,2026,"140,000",148.8017,...,역삼로 314,0,00314,00000,3122008,1,11680,11680,역삼동,
2,강남자곡아이파크,11680-4332,2014,,,27,6,2026,"82,000",74.97,...,자곡로 175,0,00175,00000,3122014,2,11680,11680,자곡동,
3,도곡렉슬,11680-3623,2006,,,6,6,2026,"155,000",84.9984,...,선릉로 221,0,00221,00000,3122006,8,11680,11680,도곡동,
4,현대13차(208~211동),11680-427,1983,,,27,6,2026,"80,000",105.31,...,압구정로29길 23,0,00023,00000,4166514,1,11680,11680,압구정동,


['aptNm', 'aptSeq', 'buildYear', 'contractTerm', 'contractType', 'dealDay', 'dealMonth', 'dealYear', 'deposit', 'excluUseAr', 'floor', 'jibun', 'monthlyRent', 'preDeposit', 'preMonthlyRent', 'roadnm', 'roadnmbcd', 'roadnmbonbun', 'roadnmbubun', 'roadnmcd', 'roadnmseq', 'roadnmsggcd', 'sggCd', 'umdNm', 'useRRRight']
